# 1.공공 데이터 포털

- 대한민국 정부가 제공하는 공공데이터 통합 플랫폼.
- 공공기관이 생성 또는 취득하여 관리하고 있는 공공데이터를 한 곳에서 제공하는 통합 창구.
- 손쉽게 공공데이터를 이용할 수 있도록 파일데이터, 오픈API, 시각화 등 다양한 방식으로 제공.
- 누구라도 쉽고 편리한 검색을 통해 원하는 공공데이터를 빠르고 정확하게 찾을 수 있다.



- 우선 xml을 딕셔너리 형태로 변환하는 라이브러리 설치

In [ ]:
# xmltodict 패키지 설치(아나콘다에는 이미 설치 됨)
# %pip install xmltodict

## 1.1.관세청_국가별 수출입실적(GW)

- 관세청 수출입통계정보의 기간, 국가, 국가코드, 수출건수,수출금액, 수입건수 등을 가져온다.

### 1.1.1.데이터 가져오기

In [ ]:
import json
import pandas as pd
import xmltodict
import urllib.request

In [ ]:
# 매개변수
key = 'YOUR_KEY'
strtYymm = 202501
endYymm = 202512
cntyCd = ''

# url 주소
url =f'https://apis.data.go.kr/1220000/nationtrade/getNationtradeList?serviceKey={key}&strtYymm={strtYymm}&endYymm={endYymm}&cntyCd={cntyCd}'

In [ ]:
# 데이터 가져오기
response = urllib.request.urlopen(url)
print(response.status)

200


In [ ]:
# xml -> dict
xml_parse = xmltodict.parse(response)               # XML 데이터를 딕셔너리로 파싱
xml_dict = json.loads(json.dumps(xml_parse))        # json 모듈을 활용해서 json 형식으로 변환하고 딕셔너리로 변환

In [ ]:
xml_dict

{'response': {'header': {'resultCode': '00', 'resultMsg': '정상서비스.'},
  'body': {'items': {'item': [{'balPayments': '3443225952',
      'expCnt': '193178',
      'expDlr': '9282566788',
      'impCnt': '1416582',
      'impDlr': '5839340836',
      'statCd': 'US',
      'statCdCntnKor1': '미국',
      'year': '2025.01'},
     {'balPayments': '-2004621045',
      'expCnt': '91388',
      'expDlr': '9204172269',
      'impCnt': '1225173',
      'impDlr': '11208793314',
      'statCd': 'CN',
      'statCdCntnKor1': '중국',
      'year': '2025.01'},
     {'balPayments': '1792245180',
      'expCnt': '57209',
      'expDlr': '4356972160',
      'impCnt': '145132',
      'impDlr': '2564726980',
      'statCd': 'VN',
      'statCdCntnKor1': '베트남',
      'year': '2025.01'},
     {'balPayments': '210300569',
      'expCnt': '34078',
      'expDlr': '2438188048',
      'impCnt': '20022',
      'impDlr': '2227887479',
      'statCd': 'TW',
      'statCdCntnKor1': '대만',
      'year': '2025.01'},
     {

In [ ]:
# 원하는 데이터 추출
# 이 경로는 API 응답 데이터의 구조에 따라 달르므로, API 문서 참고해 키를 확인해야 됨.
result = xml_dict['response']['body']['items']['item']

# 딕셔너리를 데이터프레임으로 변환
im_export = pd.json_normalize(result)
im_export.head()

,balPayments,expCnt,expDlr,impCnt,impDlr,statCd,statCdCntnKor1,year
0,3443225952,193178,9282566788,1416582,5839340836,US,미국,2025.01
1,-2004621045,91388,9204172269,1225173,11208793314,CN,중국,2025.01
2,1792245180,57209,4356972160,145132,2564726980,VN,베트남,2025.01
3,210300569,34078,2438188048,20022,2227887479,TW,대만,2025.01
4,-1223548429,146929,2363796881,344783,3587345310,JP,일본,2025.01


**데이터셋 설명**

- balPayments: 무역수지(달러)
- expCnt: 수출건수
- expDlr: 수출금액(달러)
- impCnt: 수입건수
- impDlr: 수입금액(달러)
- statCd: 국가코드
- statCdCntnKor1: 국가
- year: 기간

### 1.1.2.데이터 분석

- Pandas는 숫자가 크면 숫자를 2.805000e+03 형태로 표시한다.
- 가독성을 높이기 위해 이후 숫자 출력은 소수점 두자리를 갖는 일반 형태로 표시하게 한다.

In [ ]:
# 소수점 네 자리까지 표시
pd.set_option('display.float_format', lambda x: '%.2f' % x)

**1) 데이터 탐색 및 전처리**

- 열 이름, 데이터 형식 등 열 정보 확인

In [ ]:
im_export.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830 entries, 0 to 2829
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   balPayments     2830 non-null   str  
 1   expCnt          2830 non-null   str  
 2   expDlr          2830 non-null   str  
 3   impCnt          2830 non-null   str  
 4   impDlr          2830 non-null   str  
 5   statCd          2830 non-null   str  
 6   statCdCntnKor1  2830 non-null   str  
 7   year            2830 non-null   str  
dtypes: str(8)
memory usage: 177.0 KB


- 이후 수치 비교를 위해 숫자 데이터를 int64로 변환
- 대상 열: balPayments, expCnt, expDlr, impCnt, impDlr
- 구문 예) df['Money'].astype('int64')

In [ ]:
cols = ['balPayments', 'expCnt', 'expDlr', 'impCnt', 'impDlr']
im_export[cols] = im_export[cols].astype('int64')
im_export.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830 entries, 0 to 2829
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   balPayments     2830 non-null   int64
 1   expCnt          2830 non-null   int64
 2   expDlr          2830 non-null   int64
 3   impCnt          2830 non-null   int64
 4   impDlr          2830 non-null   int64
 5   statCd          2830 non-null   str  
 6   statCdCntnKor1  2830 non-null   str  
 7   year            2830 non-null   str  
dtypes: int64(5), str(3)
memory usage: 177.0 KB


In [ ]:
# 기초통계정보
im_export.describe()

,balPayments,expCnt,expDlr,impCnt,impDlr
count,2830.00,2830.00,2830.00,2830.00,2830.00
mean,27362076.73,4870.13,250646658.94,19270.59,223284582.21
std,477349207.86,23761.40,1105703183.57,145392.20,978743396.01
min,-2459852528.00,0.00,0.00,0.00,0.00
25%,-209850.25,16.25,556300.00,2.00,10265.75
50%,566795.50,141.00,5100655.00,21.00,1331979.50
75%,9670287.50,1117.75,56715844.00,373.75,46289235.50
max,5717645732.00,459337.00,12964960895.00,1945296.00,13729115193.00


**2) 2025년 12월 무역수지 TOP 10**

- 2025년 12월 데이터를 별도로 추출하여 **im_export12** 데이터프레임을 만듭니다.
- 무역수지(balPayments)를 기준으로 내림차순 정렬합니다.
- 정렬 결과로 어긋난 순서의 인덱스를 깔끔하게 초기화합니다.
- 상위 10개 행만 출력해 2025년 12월 무역수지 TOP 10을 확인합니다.

In [ ]:
im_export12 = im_export.loc[im_export['year']=='2025.12']
im_export12 = im_export12.sort_values(by='balPayments', ascending=False)
im_export12.reset_index(drop=True, inplace=True)

im_export12.head(10)

,balPayments,expCnt,expDlr,impCnt,impDlr,statCd,statCdCntnKor1,year
0,5561530992,232653,12335136415,1267690,6773605423,US,미국,2025.12
1,4280632585,38739,4624113700,391772,343481115,HK,홍콩,2025.12
2,3332483885,67807,6281464305,233578,2948980420,VN,베트남,2025.12
3,2256773042,68113,5131207266,25051,2874434224,TW,대만,2025.12
4,1200481846,21241,1719209860,13877,518728014,IN,인도,2025.12
5,1033101128,54471,2069320523,16861,1036219395,SG,싱가포르,2025.12
6,1008638544,43,1008679079,12,40535,MH,마샬군도,2025.12
7,896992758,33414,1203925892,6328,306933134,PH,필리핀,2025.12
8,594935421,6918,744839034,3206,149903613,TR,튀르키예,2025.12
9,439455893,16482,1050832758,8690,611376865,MX,멕시코,2025.12
